# Getting started

## 1. Convert your audio to 16 kHz mono WAV

Skipping this step is risky: a wrong sample rate (e.g. 44.1 kHz) makes Silero and WebRTC raise an error, but TEN VAD runs without one, and so does stereo audio in every model. Both give wrong results.

In [ ]:
!ffmpeg -i data/input.mp3 -ac 1 -ar 16000 data/audio_16k.wav

# -ac 1; makes it mono
# -ar 16000; sets the sample rate to 16kHz
# output is 16-bit PCM WAV

## 2. Install

In [ ]:
!uv pip install silero-vad onnxruntime numpy   # Silero
# !uv pip install ten-vad numpy                  # TEN VAD
# !uv pip install webrtcvad-wheels               # WebRTC

## 3. Find the speech segments

In [3]:
# Set audio path here
import os
import wave

AUDIO_PATH = "data/clean/clean.wav"  # or data/lotusdis-S002_T062/close.wav, data/lotusdis-S002_T062/distant.wav, or step 1's data/audio_16k.wav
AUDIO_DIR = os.path.dirname(AUDIO_PATH)  # outputs go here: segments.csv and segments/*.wav

with wave.open(AUDIO_PATH, "rb") as f:
    assert (f.getframerate(), f.getnchannels(), f.getsampwidth()) == (16000, 1, 2), "not 16 kHz mono 16-bit: convert it with step 1"

### Silero

In [5]:
import wave
import numpy as np
import torch
from silero_vad import load_silero_vad, get_speech_timestamps

INT16_SCALE = 2**15  # 16-bit samples range -32768..32767; dividing gives -1.0..1.0, the range Silero expects

with wave.open(AUDIO_PATH, "rb") as f:
    sample_rate = f.getframerate()
    pcm = f.readframes(f.getnframes())

wav = torch.from_numpy(np.frombuffer(pcm, dtype=np.int16) / INT16_SCALE).float()

model = load_silero_vad()
segments = get_speech_timestamps(wav, model, sampling_rate=sample_rate, return_seconds=True)  # default threshold is 0.5; see Tuning

segments = [(seg["start"], seg["end"]) for seg in segments]  # same (start, end) pairs as TEN VAD and WebRTC
for s, e in segments:
    print(f"{s:.2f}s -> {e:.2f}s")

0.00s -> 0.30s
1.70s -> 3.80s
4.20s -> 5.50s
5.70s -> 6.70s
6.80s -> 7.50s
7.80s -> 9.90s
10.00s -> 11.90s
12.10s -> 13.20s
13.60s -> 16.10s
16.40s -> 19.20s
19.60s -> 20.20s
20.70s -> 22.30s
22.90s -> 24.80s
25.30s -> 27.10s
28.00s -> 29.00s
29.40s -> 31.00s
31.30s -> 34.20s
34.40s -> 36.30s
36.80s -> 40.10s
40.70s -> 42.60s
42.70s -> 45.10s
45.90s -> 48.60s
49.40s -> 56.30s
56.70s -> 60.40s
60.70s -> 67.70s
69.40s -> 71.40s
72.20s -> 72.80s
73.20s -> 74.90s
75.30s -> 77.50s
78.00s -> 79.20s
79.40s -> 80.60s
81.10s -> 84.30s
84.50s -> 85.10s
85.80s -> 87.20s
87.50s -> 89.10s
89.50s -> 90.80s
91.30s -> 92.00s
92.10s -> 94.80s
95.20s -> 97.50s
97.60s -> 99.30s
100.10s -> 100.70s
101.10s -> 104.90s
105.20s -> 112.60s
112.90s -> 113.40s
114.10s -> 117.00s
117.20s -> 119.80s
120.10s -> 125.20s
125.40s -> 127.00s
127.80s -> 128.30s
128.70s -> 129.50s
129.70s -> 130.80s
131.10s -> 131.60s
132.20s -> 136.80s
137.00s -> 142.30s
142.80s -> 149.30s
150.20s -> 158.20s
158.50s -> 159.40s
159.90s -

### TEN VAD

In [ ]:
import wave
import numpy as np
from ten_vad import TenVad

HOP = 256  # samples per frame: 256 (16 ms) or 160 (10 ms)

with wave.open(AUDIO_PATH, "rb") as f:
    sample_rate = f.getframerate()
    pcm = np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16)

vad = TenVad(HOP, 0.5)  # threshold 0.5; see Tuning
segments, start = [], None
for i in range(0, len(pcm) - HOP + 1, HOP):
    t = i / sample_rate
    _, is_speech = vad.process(pcm[i:i + HOP])
    if is_speech:
        if start is None:
            start = t
    elif start is not None:
        segments.append((start, t))
        start = None
if start is not None:
    segments.append((start, len(pcm) / sample_rate))

for s, e in segments:
    print(f"{s:.2f}s -> {e:.2f}s")

### WebRTC

In [ ]:
import wave
import webrtcvad

vad = webrtcvad.Vad(3)  # mode 0 to 3
FRAME_MS = 30           # must be 10, 20 or 30

with wave.open(AUDIO_PATH, "rb") as f:
    sample_rate = f.getframerate()
    pcm = f.readframes(f.getnframes())

frame_bytes = int(sample_rate * FRAME_MS / 1000) * 2  # 2 bytes per 16-bit sample
segments, start = [], None
for i in range(0, len(pcm) - frame_bytes + 1, frame_bytes):
    t = i / 2 / sample_rate
    if vad.is_speech(pcm[i:i + frame_bytes], sample_rate):
        if start is None:
            start = t
    elif start is not None:
        segments.append((start, t))
        start = None
if start is not None:
    segments.append((start, len(pcm) / 2 / sample_rate))

for s, e in segments:
    print(f"{s:.2f}s -> {e:.2f}s")

Expected output:

```
0.00s -> 5.00s
5.10s -> 6.30s
7.20s -> 9.20s
```

## 4. Check the output by ear

The scores in this manual come from our recordings, not yours, so listen to a sample of your own output before relying on it. This check is our advice, not something we measured.

a. Save the segments next to the audio, twice: `segments.csv`, one row per segment with the audio path, start, end and a label; and `segments.txt`, a label file that Audacity (free, audacityteam.org) can open, with start, end and a label separated by tabs.

In [6]:
import csv
import os

# same columns as the reference.csv files in data/
with open(os.path.join(AUDIO_DIR, "segments.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["audio_path", "start_sec", "end_sec", "label"])
    writer.writerows((AUDIO_PATH, f"{s:.3f}", f"{e:.3f}", "speech") for s, e in segments)

# Audacity label file: start, end, label, separated by tabs, no header
with open(os.path.join(AUDIO_DIR, "segments.txt"), "w") as f:
    for s, e in segments:
        f.write(f"{s:.3f}\t{e:.3f}\tspeech\n")

b. In Audacity, open the WAV, then choose File > Import > Labels and pick `segments.txt` from the audio's folder. Each segment shows as a label under the waveform.

c. Play labels picked at random, and gaps between labels. Is each label speech, starting and ending with the words? Is each gap free of speech?

## 5. Save segments to segment WAVs

In [7]:
import os
import wave

# segments: the (start, end) pairs from step 3

with wave.open(AUDIO_PATH, "rb") as f:
    sample_rate = f.getframerate()
    pcm = f.readframes(f.getnframes())

os.makedirs(os.path.join(AUDIO_DIR, "segments"), exist_ok=True)
for i, (start, end) in enumerate(segments):
    first, last = int(start * sample_rate) * 2, int(end * sample_rate) * 2  # 2 bytes per 16-bit sample
    with wave.open(os.path.join(AUDIO_DIR, "segments", f"segment_{i:04d}.wav"), "wb") as out:
        out.setnchannels(1)
        out.setsampwidth(2)
        out.setframerate(sample_rate)
        out.writeframes(pcm[first:last])

### Optional: evaluate against a reference

Skip this step unless you have a reference: speech segments a person labeled by hand, saved as `reference.csv` in the audio's folder, with columns `start_sec` and `end_sec` (a `label` column is optional; only `speech` rows count). The sample files in `data/` come with one.

It scores `segments`, so it scores whichever model in step 3 you ran last. The metrics are the manual's:

| Metric | Meaning |
|---|---|
| FN rate | Share of reference speech the model missed: FN / (TP + FN) |
| FP rate | Share of reference non-speech the model called speech: FP / (FP + TN) |
| DCF | 0.75 × FN rate + 0.25 × FP rate; lower is better |

A 200 ms collar around each reference boundary (100 ms on each side) is not scored, because hand-placed boundaries are only accurate to a few frames. Scoring uses pyannote.metrics, as the manual's numbers do. It is already in this project's environment; elsewhere, run `uv pip install pyannote.metrics`.

In [7]:
import csv
import os
import wave
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.detection import DetectionCostFunction

REFERENCE_PATH = os.path.join(AUDIO_DIR, "reference.csv")  # columns start_sec, end_sec; a label column is optional
COLLAR = 0.2  # seconds around each reference boundary that are not scored: 100 ms on each side

assert os.path.exists(REFERENCE_PATH), f"no reference at {REFERENCE_PATH}: label one in Audacity, or skip this step"

with open(REFERENCE_PATH, newline="") as f:
    reference_segments = [
        (float(row["start_sec"]), float(row["end_sec"]))
        for row in csv.DictReader(f)
        if row.get("label", "speech") == "speech"
    ]

with wave.open(AUDIO_PATH, "rb") as f:
    duration = f.getnframes() / f.getframerate()


def to_annotation(segs):
    """Turn (start, end) pairs into the speech annotation pyannote.metrics scores."""
    annotation = Annotation()
    for s, e in segs:
        annotation[Segment(s, e)] = "speech"
    return annotation


# DCF = 0.75 x FN rate + 0.25 x FP rate, as NIST defines it
metric = DetectionCostFunction(collar=COLLAR)
uem = Timeline([Segment(0, duration)])  # score the whole file, so silence at either end counts as non-speech
c = metric(to_annotation(reference_segments), to_annotation(segments), uem=uem, detailed=True)

fn_rate = c["miss"] / c["positive class total"]           # share of speech missed
fp_rate = c["false alarm"] / c["negative class total"]    # share of non-speech called speech

print(f"FN rate {fn_rate:.3f}, FP rate {fp_rate:.3f}, DCF {c[metric.name]:.3f}")

FN rate 0.008, FP rate 0.076, DCF 0.025
